# 🧠📍 Smart Tagging

Metrica's **automatic** tactical annotation of the tactical video: phases of play and set
pieces, each one a time interval with a code and a set of tags.

This notebook covers loading it, what is in it, and the two things that will bite you if you
treat it as ground truth.

---

## Loading

The file is a **SportsCode XML**, the standard format in football performance analysis, so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/sportscode/) reads it directly.

`games.asset_paths()` builds the path, so nothing here spells a filename out and
`MTAH_DATA_DIR` keeps working if you store the bundle elsewhere.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from kloppy import sportscode

import games

path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
print('%d instances' % len(events))
events.head()

638 instances


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,NaN,NaN


## What is in it

Each row is one tagged interval:

- **timestamp / end_timestamp** — start and end, as pandas timedeltas from the first frame
- **code** — the phase or set piece, upper case (`SET PIECES`, `BUILD UP`, …)
- **Team** — which side the phase belongs to
- **Half** — `1st Half` or `2nd Half`, as strings
- plus per-code tags: `Type`, `Side`, `Max Players in the box`, `Direction of ball entry`

Note the capital in `Team`: kloppy passes the tag groups through with the names the XML uses, so
every tag column is title case. The full vocabulary is in [DATA.md](../DATA.md).

In [2]:
events['code'].value_counts()

code
PLAYERS IN THE BOX              97
BALL IN FINAL THIRD             93
BALL IN THE BOX                 72
PROGRESSION                     66
DEFENDING IN MIDDLE THIRD       66
SET PIECES                      59
CREATING CHANCES                30
DEFENDING IN DEFENSIVE THIRD    30
LONG BALL                       29
ATTACKING TRANSITION            24
DEFENSIVE TRANSITION            24
BUILD UP                        22
DEFENDING IN ATTACKING THIRD    22
GOALS                            4
Name: count, dtype: int64

### Codes are upper case, tag values are title case

`SET PIECES` and `Kick Off (Start)`. Match case-insensitively unless you enjoy silent empty
results.

In [3]:
set_pieces = events[events['code'].str.lower() == 'set pieces']
set_pieces['Type'].value_counts()

Type
Throw-in                 26
Corner Kick              12
Goal Kick                10
Free Kick                 5
Kick Off (After Goal)     4
Kick Off (Start)          2
Name: count, dtype: int64

## Converting to video frames

The timestamps are **seconds from the first frame of the tactical video**, so a video frame —
and therefore a submission row — is just `round(seconds * 25)`. There is no offset to discover.

`starter/load_smart_tagging.py` does this without kloppy, and re-checks the time base against
both kickoffs every time it runs.

In [4]:
FPS = games.GAMES[games.GAME]['fps']

frames = pd.DataFrame({
    'code': events['code'],
    'Team': events['Team'],
    'start_frame': (events['timestamp'].dt.total_seconds() * FPS).round().astype(int),
    'end_frame': (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int),
})
frames.head()

,code,Team,start_frame,end_frame
0,SET PIECES,N/A,11097,11222
1,SET PIECES,N/A,49631,49756
2,SET PIECES,N/A,62012,62137
3,SET PIECES,N/A,75457,75582
4,SET PIECES,N/A,113660,113785


## Two things to be careful about

### 1. `Team` is not always a team

36 instances are tagged `N/A` — including **every throw-in**, which are exactly the restarts
where knowing the side would be worth the most. Handle it explicitly; do not let it fall
through to a default.

In [5]:
events['Team'].value_counts(dropna=False)

Team
Barcelona       314
Atletic Club    288
N/A              36
Name: count, dtype: int64

### 2. An interval is a window, not an instant

A set piece is tagged as a span of a few seconds *around* the restart, not at the frame the ball
is struck. The two `Kick Off (Start)` tags below both **contain** the real kickoff rather than
starting on it — they open about 3.5 seconds early.

So `start_frame` is not the restart frame, and the same imprecision applies to every other
interval in the file. Use a tag to say *roughly when* something happened, and the tracking to say
exactly when.

In [6]:
kickoffs = set_pieces[set_pieces['Type'].str.lower() == 'kick off (start)'].copy()
kickoffs['start_frame'] = (kickoffs['timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['end_frame'] = (kickoffs['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['real_kickoff'] = [11185, 113742]          # from the ground truth
kickoffs['contains_it'] = ((kickoffs['start_frame'] <= kickoffs['real_kickoff'])
                           & (kickoffs['real_kickoff'] <= kickoffs['end_frame']))

kickoffs[['Half', 'start_frame', 'end_frame', 'real_kickoff', 'contains_it']]

,Half,start_frame,end_frame,real_kickoff,contains_it
0,1st Half,11097,11222,11185,True
4,2nd Half,113660,113785,113742,True


---

## Why this is worth using anyway

Every phase carries a **team** tag, and every possession has a team. Every set piece is a
restart, and every restart ends a dead-ball stretch. The correspondence is not exact — that is
the challenge — but it is a strong prior, for free, on both halves of the label you are asked
to produce.

Just remember the intervals are *phases*, not possessions: a Build Up can span one possession,
stop short of one, or cover two.

**Next:** [`02_tracking_atd.ipynb`](02_tracking_atd.ipynb) for the tracking data, then
[`03_first_submission.ipynb`](03_first_submission.ipynb) to build and score a submission.

## Findings: what Smart Tagging tells us, and how we use it

**What we measured**
- 638 tagged intervals, 14 codes. 59 are `SET PIECES`: 26 throw-ins, 12 corners, 10 goal kicks, 5 free kicks, 4 kick-offs after a goal, 2 start kick-offs.
- Team split is nearly even (Barcelona 314, Athletic Club 288), with 36 `N/A`. All 26 throw-ins are `N/A`, so the tags can't tell us who restarts them.
- Set-piece windows are 125 frames (5 s) long. In both kick-offs the real kickoff falls inside the window, about 3.3–3.5 s after it opens (88 and 82 frames in). The window is not centred on the event.

**What it means for the task**
1. **Set pieces mark where dead ball (`D`) hands back to live play.** There are 59 of them, so they give us candidate segment boundaries, and `spellF1` rewards getting boundaries right.
2. **Phase codes give a team prior.** They tell us roughly who has the ball, but a phase is not a possession. Use them to break ties where tracking is ambiguous, not as labels.
3. **Tags are coarse, tracking is exact.** Use a tag to say roughly *when*, and the tracking (ball speed, ball-to-player distance) to say exactly *when*.

**What we don't know yet (to verify before relying on it)**
- The kick-off offset comes from only 2 samples. Check whether other set-piece types show the same offset in the public 5 minutes.
- We can only validate against the first 5 minutes of ground truth, so a rule that looks great there may not generalise.
- Which side is Home vs Away in the `H`/`A` labels (check `DATA.md` before mapping `Team` onto labels).

How this helps the team win

The ranking metric is spellF1, which scores whole possessions recovered. The baseline's frame-level accuracy barely changes with smoothing, but spellF1 goes from 0.04 to 0.42 on the public 5 minutes. So fragmentation is the main thing hurting the score, and the tags help with that in three ways:

- Fewer false splits. The baseline is "nearest player to ball plus a majority vote". Smart Tagging gives an independent signal for where a possession should start or end. Smoothing hard inside a phase and allowing changes only near boundaries should raise spellF1 without hurting macroF1.
- Better D detection. Dead-ball stretches are a whole class of the label, and the set-piece windows tell you where to look for them. Use them to gate a "ball stationary and no player near" rule.
- Tie-breaking when the ball is invisible. The ball is missing on 41% of scored frames. When tracking has nothing to say, the phase's team tag is a free fallback, much better than carrying the last label forward.

Suggested next steps

1. Measure how often each set-piece type's window start or end lines up with a ground-truth D→H/A change in the public 5 minutes. That tells us whether to trust the windows as boundaries.
2. Measure how often the phase Team tag agrees with ground-truth possession. That decides whether it works as a prior or only as a tie-break.
3. Add the tag-derived features to the smoothing logic in baseline.py and compare spellF1 before and after.

---

## Next steps: can we trust the tags?

You have the true answers for the first 5 minutes of the match (the "answer key"). The tags are Metrica's automatic guesses. Each step below checks how well the guesses line up with the answer key.

### Step 1: Do set-piece tags really mark the end of a dead ball?

**Question:** when a tag says "throw-in happens here", does the true label switch from `D` (dead ball) to `H`/`A` inside that window?

**How:**
1. Take every set-piece tag in the first 5 minutes.
2. Find the frame where the answer key changes from `D` to live play.
3. Check whether that frame sits inside the tag's 5-second window, and where in the window (early, middle, late).

**Why it matters:** the 2 kick-offs fall about 3.4 s into their windows. If throw-ins, goal kicks and corners show the same pattern, we can predict the restart frame closely (window start + ~3.4 s). If not, we only treat the window as "somewhere around here".

- **Good result:** most tags contain the real restart, at a consistent point in the window.
- **Bad result:** many tags miss it, so we use them lightly or not at all.

### Step 2: Does the tag's team match who actually has the ball?

**Question:** when a phase is tagged "Barcelona", does the answer key show Barcelona in possession?

**How:**
1. For each frame in the first 5 minutes, look up the phase tag covering it.
2. Compare the tag's team to the true label.
3. Count the matches (for example "agrees on 80% of live frames").

**Why it matters:** the ball is invisible on 41% of scored frames, and there the tracking rule has nothing to work with. If the tag agrees often enough, it becomes a decent fallback.

- **Good result:** high agreement (around 75-80% or more), so we use the tag as a backup when the ball is missing.
- **Bad result:** barely better than a coin flip (50%), so the tag is no help.

We also need to map `Barcelona` / `Athletic Club` onto `H` / `A` (see [DATA.md](../DATA.md)).

### Step 3: Put it into the model and see if the score improves

**Question:** does using what we learned in steps 1 and 2 raise the score?

**How:**
1. Start from the baseline (nearest player to the ball, with smoothing): `spellF1` 0.4242 on the public 5 minutes.
2. Add one idea at a time:
   - allow possession to change near the end of a set-piece window, and be stricter elsewhere
   - use the tag's team when the ball is not visible
3. Re-score after each change and keep only the changes that help.

**Why it matters:** the final ranking is `spellF1`, which rewards getting whole possessions right. Tags are only useful if they raise that number.

### Caution

We only have 5 minutes of answer key but are scored on the whole match. Prefer simple rules backed by clear evidence over complicated ones tuned to this small sample; the complicated ones tend to score worse on the private test.
